# BINN-exact equivalence checks (layered MPNN vs masked FFNN)

This notebook is intentionally minimal:

- It **builds a layered message-passing schedule** from the saved padded graph.
- It runs a **numerical equivalence check** between two implementations of the same computation:
  - index-add message passing (GNN view)
  - sparse masked matrix multiplication (FFNN view)
- The *canonical* version of these checks lives in `tests/` and should be run with `pytest`.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/binn_gnn_repo_ready")

sys.path.append(str(PROJECT_ROOT / "src"))

import os
os.environ["BINN_GNN_BASE"] = str(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

Mounted at /content/drive
Project root: /content/drive/MyDrive/binn_gnn_repo_ready


In [2]:
!pip install torch torchvision torchaudio
!pip install torch-geometric
!pip install networkx pandas numpy scikit-learn tqdm
!pip install pytest

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.1 MB/s eta 0:00:00


In [3]:
# ==== 0) Paths ====
from pathlib import Path
import os

BASE_DIR = Path(os.environ.get("BINN_GNN_BASE", ".")).resolve()
OUT_DIR  = BASE_DIR / "outputs"
LAYERED_GRAPH_DIR = OUT_DIR / "graph_layered_binn"

print("BASE_DIR:", BASE_DIR)
print("LAYERED_GRAPH_DIR:", LAYERED_GRAPH_DIR)

required = [
    LAYERED_GRAPH_DIR / "node_table_layered.csv",
    LAYERED_GRAPH_DIR / "edge_table_layered.csv",
    LAYERED_GRAPH_DIR / "edge_index_layered.pt",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required layered graph artifacts:\n" + "\n".join(missing))

BASE_DIR: /content/drive/MyDrive/binn_gnn_repo_ready
LAYERED_GRAPH_DIR: /content/drive/MyDrive/binn_gnn_repo_ready/outputs/graph_layered_binn


In [4]:
# ==== 1) Build the layered propagation schedule ====
import numpy as np
import pandas as pd
import torch

from binn_gnn.graph.schedule import build_layered_schedule

node_table = pd.read_csv(LAYERED_GRAPH_DIR / "node_table_layered.csv")
edge_table = pd.read_csv(LAYERED_GRAPH_DIR / "edge_table_layered.csv")
edge_index = torch.load(LAYERED_GRAPH_DIR / "edge_index_layered.pt").long()

node_layer = node_table["layer"].to_numpy(dtype=np.int64)
edge_type = edge_table["edge_type"].astype(str).tolist()

gene_ids = node_table.query("node_type=='gene' and layer==0")["layered_id"].astype(int).tolist()
root_ids = node_table.query("node_type=='pathway'")                     .groupby("orig_id")["layered_id"].max()                     .tolist()

# Use roots saved by the layerization step, if available
root_path = LAYERED_GRAPH_DIR / "root_pathway_idx.pt"
if root_path.exists():
    root_ids = torch.load(root_path).long().tolist()

schedule = build_layered_schedule(
    edge_index=edge_index,
    node_layer=node_layer,
    edge_type=edge_type,
    gene_ids=gene_ids,
    root_ids=root_ids,
)

print("Schedule | N:", schedule.N, "E:", schedule.E, "L:", schedule.L, "roots:", schedule.root_ids.numel())
print("Padding edges:", int(schedule.is_padding_edge.sum().item()))

Schedule | N: 16283 E: 44955 L: 12 roots: 29
Padding edges: 2082


In [5]:
# ==== 2) Numerical equivalence check on random inputs ====
# This mirrors the unit test in `tests/test_equivalence_layered.py`,
# but runs on the *real* layered graph.

def layered_forward_index_add(h0, schedule, edge_weight, node_bias, learn_padding=False):
    h = h0.clone()
    for step in schedule.steps:
        if step.eid.numel() == 0:
            continue
        src = step.src.to(h.device)
        dst_unique = step.dst_unique.to(h.device)
        dst_pos = step.dst_pos.to(h.device)
        eid = step.eid.to(h.device)

        w = edge_weight.index_select(0, eid)
        if not learn_padding:
            pad_mask = schedule.is_padding_edge.index_select(0, eid).to(h.device)
            if pad_mask.any():
                w = w.clone()
                w[pad_mask] = 1.0

        msg = h.index_select(1, src) * w.view(1, -1)
        agg = torch.zeros((h.size(0), dst_unique.numel()), device=h.device, dtype=h.dtype)
        agg.index_add_(1, dst_pos, msg)

        h[:, dst_unique] = torch.tanh(agg + node_bias.index_select(0, dst_unique).view(1, -1))
    return h

def masked_ffnn_forward_sparsemm(h0, schedule, edge_weight, node_bias, learn_padding=False):
    h = h0.clone().t().contiguous()  # [N,B]
    N = h.size(0)

    for step in schedule.steps:
        if step.eid.numel() == 0:
            continue
        src = step.src
        dst = step.dst
        dst_unique = step.dst_unique
        eid = step.eid

        w = edge_weight.index_select(0, eid)
        if not learn_padding:
            pad_mask = schedule.is_padding_edge.index_select(0, eid)
            if pad_mask.any():
                w = w.clone()
                w[pad_mask] = 1.0

        idx = torch.stack([dst, src], dim=0)
        W = torch.sparse_coo_tensor(idx, w, size=(N, N), dtype=h.dtype)
        agg = torch.sparse.mm(W, h)  # [N,B]

        upd = torch.tanh(agg.index_select(0, dst_unique) + node_bias.index_select(0, dst_unique).view(-1, 1))
        h.index_copy_(0, dst_unique, upd)

    return h.t().contiguous()

torch.manual_seed(0)

B = 4
N = schedule.N
E = schedule.E

h0 = torch.randn((B, N), dtype=torch.float64)
edge_weight = torch.randn((E,), dtype=torch.float64)
node_bias = torch.randn((N,), dtype=torch.float64)

h_a = layered_forward_index_add(h0, schedule, edge_weight, node_bias, learn_padding=False)
h_b = masked_ffnn_forward_sparsemm(h0, schedule, edge_weight, node_bias, learn_padding=False)

max_abs = (h_a - h_b).abs().max().item()
print("max |Δ|:", max_abs)
print("allclose:", torch.allclose(h_a, h_b, atol=1e-10, rtol=1e-10))

max |Δ|: 3.164135620181696e-15
allclose: True


## 3) Run the unit tests

In a cloned repo (local or Colab), run:

```bash
pytest -q
```

This is the authoritative check, because it will run the toy-graph equivalence test and other invariants.


In [7]:
%cd /content/drive/MyDrive/binn_gnn_repo_ready
!pip -q install -e .
!pytest -q

/content/drive/MyDrive/binn_gnn_repo_ready
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for binn-gnn (pyproject.toml) ... done
...                                                                      [100%]


In [8]:
%cd /content/drive/MyDrive/binn_gnn_repo_ready
!pytest -q tests

/content/drive/MyDrive/binn_gnn_repo_ready
...                                                                      [100%]


In [6]:
# (Optional) Run unit tests from inside a cloned repo:
!pytest -q


==================================== ERRORS ====================================
_ ERROR collecting drive/MyDrive/binn_gnn_refactor/tests/test_depth_schedule.py _
ImportError while importing test module '/content/drive/MyDrive/binn_gnn_refactor/tests/test_depth_schedule.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.12/importlib/__init__.py:90: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
drive/MyDrive/binn_gnn_refactor/tests/test_depth_schedule.py:4: in <module>
    from binn_gnn.graph.schedule import build_depth_schedule
E   ModuleNotFoundError: No module named 'binn_gnn'
_ ERROR collecting drive/MyDrive/binn_gnn_refactor/tests/test_directed_swap.py _
ImportError while importing test module '/content/drive/MyDrive/binn_gnn_refactor/tests/test_directed_swap.py'.
Hint: make sure your test modules/packages have valid Python names.
Tra